# 第 7 章习题与解答

> 本章习题聚焦 greedy decoding 实现、KV cache 复杂度分析和采样策略组合。建议先自己思考,再展开答案。

## Exercise 7.1(易)手写 greedy decoding 生成 20 个 token

**题目**:不使用任何 sampling 策略,用 pure greedy 方式写一个生成函数,生成恰好 20 个 token。要求:
1. 不使用 KV cache(朴素版)
2. 每步打印当前选中的 token id

提示:`torch.argmax(logits[:, -1, :], dim=-1)`

<details><summary><b>参考答案</b></summary>

```python
import torch

def greedy_generate(model, input_ids, num_tokens=20):
    """朴素 greedy decoding,无 KV cache。"""
    generated = input_ids.clone()
    for step in range(num_tokens):
        with torch.inference_mode():
            outputs = model(generated)
        # 取最后一个位置的 logits,argmax 选 token
        next_logits = outputs.logits[:, -1, :]
        next_token = torch.argmax(next_logits, dim=-1, keepdim=True)
        print(f"  step {step}: token_id={next_token.item()}")
        generated = torch.cat([generated, next_token], dim=-1)
    return generated

# 调用
# result = greedy_generate(model, input_ids, num_tokens=20)
```

**关键点**:
1. `outputs.logits[:, -1, :]` —— 永远取最后位置(因为前面的已经确定了)
2. `torch.argmax(dim=-1, keepdim=True)` —— keepdim 保持 (batch, 1) 形状,方便 cat
3. 不用 `do_sample`,纯 argmax —— **输出完全确定**,同一 prompt 永远生成同一序列

**对比 minimind**:`do_sample=False` 时,minimind 第 335 行正是 `torch.argmax(logits, dim=-1, keepdim=True)`,等价于本题实现。

</details>

## Exercise 7.2(中)KV cache 为什么把复杂度从 O(T²) 降到 O(T)?

**题目**:解释为什么朴素自回归生成是 $O(T^2)$,而使用 KV cache 后是 $O(T)$。请从**每步 forward 的 token 数**角度定量分析,并说明为什么缓存 K/V 是安全的(K 和 V 不会变)。

<details><summary><b>参考答案</b></summary>

### 朴素版:O(T²)

生成 $T$ 个 token,每步 forward 的序列长度递增:

$$\text{总计算量} \propto \sum_{t=1}^{T} (L_0 + t) = T \cdot L_0 + \frac{T(T+1)}{2} = O(T^2)$$

其中 $L_0$ 是 prompt 长度。每一步都对**所有历史 token** 重新计算 K、V、Q 和注意力。

### KV cache 版:O(T)

每步只 forward **1 个新 token**,历史 K/V 从缓存读取:

$$\text{总计算量} \propto L_0 + \underbrace{1 + 1 + \cdots + 1}_{T \text{次}} = L_0 + T = O(T)$$

### 为什么缓存 K/V 是安全的?

在自回归生成中,**历史 token 的输入不会变**。对于位置 $i$ 的 token:

- 它的 embedding $x_i$ 是固定的(token id 不变)
- $K_i = x_i W_K$,$V_i = x_i W_V$ —— $W_K, W_V$ 是模型参数(推理时不变)
- 所以 **$K_i$ 和 $V_i$ 只依赖于 $x_i$,而 $x_i$ 已经确定了**

唯一变化的是 Q(因为新 token 的 Q 要和历史 K 做点积),但 Q 不需要缓存 —— 只算当前新 token 的 Q 即可。

**数学证明**:注意力分数 $\text{attn}_i = \text{softmax}(Q_{\text{new}} \cdot K_i / \sqrt{d})$。$K_i$ 可以缓存,$Q_{\text{new}}$ 每步重新算(只有 1 个)。

### 实际加速

| 生成长度 | 朴素 forward 次数 | cache forward 次数 | 加速比 |
|---|---|---|---|
| 50 | ~1300 | ~53 | ~25× |
| 200 | ~20000 | ~203 | ~100× |
| 1000 | ~500000 | ~1003 | ~500× |

> 这也是为什么 `use_cache=True` 是 minimind 的默认值 —— 关掉它生成速度会慢一到两个数量级。

</details>

## Exercise 7.3(难)top_k=50 和 top_p=0.9 同时设置,哪个先生效?

**题目**:在 minimind 的 `generate` 中,`top_k=50` 和 `top_p=0.9` 可以同时生效。请分析:

1. 代码中两者的**执行顺序**是什么?
2. 假设词表有 6400 个 token,当前 logits 排序后前 50 个累积概率已达 0.95。最终保留多少个 token?
3. 如果调换 top_k 和 top_p 的执行顺序,结果会不同吗?

<details><summary><b>参考答案</b></summary>

### 1. 执行顺序

查看 `model_minimind.py` 第 328-334 行:

```python
if top_k > 0:                                           # 先执行 top_k
    logits[logits < torch.topk(logits, top_k)[0][..., -1, None]] = -float('inf')
if top_p < 1.0:                                         # 后执行 top_p
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    mask = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1) > top_p
    ...
```

**top_k 先,top_p 后**。两者是**串联(sequential)**的,不是并行取交集。

### 2. 具体场景分析

条件:vocab=6400,top_k=50,top_p=0.9,前 50 个累积概率=0.95。

**执行 top_k(第 1 步)**:
- 从 6400 个 token 砍到 50 个(设其余为 $-\infty$)
- 此时只有 50 个 token 有非零概率

**执行 top_p(第 2 步)**:
- 对剩余 50 个 token 重新排序 + softmax + 累积
- 累积到 0.9 时截断:由于前 50 个累积=0.95,可能在第 ~40 个就达到 0.9
- 最终保留约 **40 个 token**(0.9 < 0.95)

**结论**:最终保留约 40 个 token(top_p 进一步缩小了 top_k 的候选集)。

### 3. 调换顺序会不同吗?

**会不同,但通常差异不大**。

如果先 top_p 后 top_k:

```python
# 假设性调换
if top_p < 1.0:     # 先:从 6400 个中按累积概率砍到 ~某数量
    ...
if top_k > 0:       # 后:再砍到固定 50 个
    ...
```

- 如果 top_p 保留的 token 数 > 50:top_k 再砍到 50,结果和原来一样
- 如果 top_p 保留的 token 数 < 50:top_k 不起作用(已经少于 50 了),结果**比原来更少**

**minimind 选择 top_k 先的原因**:先做固定数量截断(快,$O(n \log k)$),再做概率累积截断(在更小的候选集上排序,更快)。这是**效率优先**的设计。

> **实践建议**:不要同时设过小的 top_k 和过小的 top_p —— 两者叠加会过度截断,导致采样空间太小,输出失去多样性。通常 top_k=0(禁用)或 top_k 设大,主要靠 top_p 控制质量。

</details>